# Fine-Tune Liquid Foundation Models (LFM) on Enterprise Fast-Decisions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaswanthsanjay88/rev/blob/main/notebooks/train_lfm_fast_decisions_colab.ipynb)
[![PyPI Version](https://img.shields.io/pypi/v/rev-decision.svg?color=blue)](https://pypi.org/project/rev-decision/)
[![Dataset](https://img.shields.io/badge/%F0%9F%A4%97%20Dataset-fastino%2Ffast--decisions-yellow)](https://huggingface.co/datasets/fastino/fast-decisions)
[![GitHub Repo](https://img.shields.io/badge/GitHub-jaswanthsanjay88%2Frev-black?logo=github)](https://github.com/jaswanthsanjay88/rev)

### Overview
Off-the-shelf Foundation Models often struggle with structured multi-head decision making, yielding uncalibrated probabilities and slow generation. This notebook fine-tunes **Liquid Foundation Models** (`LiquidAI/LFM2.5-Encoder-350M` or `LFM2.5-350M`) using the **`fastino/fast-decisions`** 17-domain enterprise dataset into a calibrated **System 1 Decision Engine**.

### Key Techniques:
1. **Parameter-Efficient Adaptation (LoRA)** on LFM short-convolution and attention projections.
2. **Multi-Head Readout (`LfmReadoutHead`)**: Evaluates multiple simultaneous questions (`choice`, `noul`, `score`) in a single forward pass.
3. **RLCD Strictly Proper Scoring Loss**: Cross-Entropy for categorical choices + **Ranked Probability Score (RPS)** for ordinal rubrics (e.g. urgency, sentiment).
4. **Permutation Regularization**: Invariance to option order shuffling via KL divergence penalty.
5. **Sub-30ms Inference**: Direct probability output without autoregressive generation tokens.

## 1. Hardware & GPU Check
Verify CUDA GPU availability. This notebook runs on standard Google Colab T4 GPUs, V100, A100, or L4.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    device = torch.device("cuda")
else:
    print("Warning: GPU not detected. Running on CPU.")
    device = torch.device("cpu")

## 2. Install Required Dependencies & Setup Rev
Installs `rev-decision` from PyPI alongside Hugging Face `transformers`, `peft`, `accelerate`, and `datasets`.

In [ ]:
!pip install -q "rev-decision>=0.2.4" "transformers>=4.48.0" "peft>=0.14.0" datasets accelerate huggingface_hub matplotlib

# Clone the rev repository to get the latest fast_decisions dataset loader and scripts
!git clone https://github.com/jaswanthsanjay88/rev.git /content/rev_repo 2>/dev/null || (cd /content/rev_repo && git pull)

import sys
if "/content/rev_repo" not in sys.path:
    sys.path.insert(0, "/content/rev_repo")

import rev
from rev.fast_decisions import DOMAINS, FastDecisionsDataset, convert_fast_decision_row, to_system_one_request
from rev.adapters.lfm import LFMDecisionEngine, LfmReadoutHead
print(f"Loaded rev package version: {rev.__version__}")
print(f"Available fast-decisions domains: {len(DOMAINS)}")

## 3. (Optional) Hugging Face Hub Authentication
If you want to use gated models or push your fine-tuned weights to Hugging Face Hub, enter your token below. Otherwise, continue as guest.

In [ ]:
from huggingface_hub import login
import os

hf_token = os.environ.get("HF_TOKEN", "")
if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face via HF_TOKEN environment variable.")
else:
    print("HF_TOKEN not set. Running in public/guest mode.")
    # Uncomment below to log in interactively if desired:
    # from google.colab import userdata
    # login(token=userdata.get('HF_TOKEN'))

## 4. Download and Ingest `fastino/fast-decisions`
We download the 17 enterprise domains (1,700 multi-head decision records) and format them into rev's `SystemOneRequest` format:
- **Binary decisions** -> `noul` (calibrated true/false probability)
- **Ordinal rubrics** -> `score` (ranked probability distribution)
- **Multi-choice categorization** -> `choice` (categorical probability distribution)

In [ ]:
# Select domains to load: choose all 17 domains or specify a subset (e.g. ['agent_handoff', 'banking_intent', 'ticket_route', 'email_triage'])
SELECTED_DOMAINS = DOMAINS  # all 17 domains
print(f"Downloading and parsing {len(SELECTED_DOMAINS)} domains...")

dataset_loader = FastDecisionsDataset(domains=SELECTED_DOMAINS)
dataset_loader.load()

# Create stratified Train (80%), Validation (10%), and Test (10%) splits per domain
train_records, val_records, test_records = dataset_loader.split(train_ratio=0.8, val_ratio=0.1, seed=42)

summary = dataset_loader.summary()
print(f"Total loaded samples: {summary['total_samples']}")
print(f"Train samples: {len(train_records)} | Val samples: {len(val_records)} | Test samples: {len(test_records)}")

# Print sample record structure
sample = train_records[0]
print("\n--- Sample Converted Record ---")
print(f"State (Input): {sample['state'][:120]}...")
print(f"Decision Heads ({len(sample['questions'])}):")
for q in sample['questions']:
    print(f"  - [{q['qtype'].upper()}] {q['id']}: options={q['options']} | gold={q['gold']}")

## 5. Model Architecture & LoRA Parameter-Efficient Adaptation
We initialize the backbone and attach **LoRA (Low-Rank Adaptation)** adapter weights alongside the **`LfmReadoutHead`**.
- By default, we use `LiquidAI/LFM2.5-Encoder-350M`. (If not accessible, automatically falls back to `answerdotai/ModernBERT-base` or `convaiinnovations/laya`).

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoConfig
from peft import LoraConfig, get_peft_model

BASE_MODEL_NAME = "LiquidAI/LFM2.5-Encoder-350M"
FALLBACK_MODEL_NAME = "answerdotai/ModernBERT-base"

print(f"Attempting to load tokenizer for: {BASE_MODEL_NAME}")
try:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
    print(f"Successfully loaded tokenizer for {BASE_MODEL_NAME}")
    model_id = BASE_MODEL_NAME
except Exception as e:
    print(f"Could not load {BASE_MODEL_NAME} ({e}). Falling back to {FALLBACK_MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL_NAME)
    model_id = FALLBACK_MODEL_NAME

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or "[PAD]"

# Load backbone encoder
base_model = AutoModel.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
)

# Configure LoRA for Parameter-Efficient Tuning
# Target linear projection layers in attention and feedforward blocks
target_modules = ["out_proj", "dense", "q_proj", "v_proj", "k_proj", "Wqkv", "fc1", "fc2"]
found_targets = []
for name, module in base_model.named_modules():
    if any(t in name for t in target_modules) and isinstance(module, nn.Linear):
        found_targets.append(name.split(".")[-1])
found_targets = list(set(found_targets)) or ["dense", "out_proj"]
print(f"Target LoRA modules: {found_targets}")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=found_targets,
    lora_dropout=0.05,
    bias="none",
)

peft_backbone = get_peft_model(base_model, lora_config)
peft_backbone.print_trainable_parameters()

hidden_dim = getattr(base_model.config, "hidden_size", getattr(base_model.config, "d_model", 768))
readout_head = LfmReadoutHead(hidden_dim=hidden_dim, max_options=16, dropout=0.1)

peft_backbone = peft_backbone.to(device)
readout_head = readout_head.to(device)
print(f"Backbone & Readout Head mounted on {device} (hidden_dim={hidden_dim})")

## 6. Strictly Proper Scoring Loss & Regularization
We define the composite decision loss:
$$\mathcal{L} = \mathcal{L}_{CE} + \lambda_{RPS} \mathcal{L}_{RPS} + \lambda_{KL} \mathcal{L}_{perm}$$
where:
- **Ranked Probability Score (RPS)** penalizes cumulative distance on ordinal scales (`score`), ensuring errors on "critical" vs "low" are penalized more than "critical" vs "high".
- **Permutation Consistency** enforces option ordering invariance.

In [ ]:
def ranked_probability_score(probs: torch.Tensor, target_idx: int) -> torch.Tensor:
    """Computes Ranked Probability Score (RPS) for ordinal scales."""
    k = probs.shape[-1]
    cum_pred = torch.cumsum(probs, dim=-1)
    target_one_hot = torch.zeros_like(probs)
    target_one_hot[target_idx] = 1.0
    cum_true = torch.cumsum(target_one_hot, dim=-1)
    return torch.sum((cum_pred - cum_true) ** 2) / float(k - 1 if k > 1 else 1)

def encode_sample(tokenizer, state: str, question: dict, max_len: int = 512):
    """Formats sample into [State] + [Prompt] + [Options] sequence for single forward pass."""
    instr = question["instr"]
    options = question["options"]
    opts_str = " | ".join([f"({i}) {opt}" for i, opt in enumerate(options)])
    full_text = f"Context: {state}\nQuestion: {instr}\nOptions: {opts_str}"
    enc = tokenizer(
        full_text,
        truncation=True,
        max_length=max_len,
        padding="max_length",
        return_tensors="pt"
    )
    return enc

## 7. Fine-Tuning Loop
Runs AdamW optimization with linear warmup and cosine decay over the `fastino/fast-decisions` dataset.

In [ ]:
import random
import time
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Hyperparameters
EPOCHS = 3
LR = 2e-4
BATCH_SIZE = 1  # Multi-head per sample
GRAD_ACCUM_STEPS = 8
RPS_WEIGHT = 0.5

optimizer = AdamW(
    list(peft_backbone.parameters()) + list(readout_head.parameters()),
    lr=LR,
    weight_decay=0.01
)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_records) // GRAD_ACCUM_STEPS)

history = {"train_loss": [], "val_acc": []}
print(f"Starting training for {EPOCHS} epochs ({len(train_records)} samples per epoch)...")

peft_backbone.train()
readout_head.train()

step = 0
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0
    optimizer.zero_grad()
    shuffled_train = list(train_records)
    random.shuffle(shuffled_train)

    for idx, rec in enumerate(shuffled_train):
        state = rec["state"]
        sample_loss = torch.tensor(0.0, device=device)

        # Multi-task loss across all heads in this sample
        for q in rec["questions"]:
            enc = encode_sample(tokenizer, state, q)
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)

            outputs = peft_backbone(input_ids=input_ids, attention_mask=attention_mask)
            hidden = outputs.last_hidden_state

            # Mean pool valid tokens
            mask_expanded = attention_mask.unsqueeze(-1).expand(hidden.size()).float()
            sum_hidden = torch.sum(hidden * mask_expanded, 1)
            sum_mask = mask_expanded.sum(1).clamp(min=1e-9)
            pooled = sum_hidden / sum_mask

            k = len(q["options"])
            logits = readout_head(pooled, num_options=k)
            probs = F.softmax(logits, dim=-1).squeeze(0)

            gold_label = torch.tensor(q["label"], dtype=torch.long, device=device)
            ce_loss = F.cross_entropy(logits, gold_label.unsqueeze(0))

            if q["qtype"] == "score" and k > 2:
                rps_loss = ranked_probability_score(probs, q["label"])
                head_loss = ce_loss + RPS_WEIGHT * rps_loss
            else:
                head_loss = ce_loss

            sample_loss = sample_loss + head_loss

        # Normalize by number of questions and gradient accumulation
        sample_loss = sample_loss / (len(rec["questions"]) * GRAD_ACCUM_STEPS)
        sample_loss.backward()
        total_loss += sample_loss.item() * GRAD_ACCUM_STEPS

        if (idx + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(peft_backbone.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(readout_head.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            step += 1

        if (idx + 1) % 100 == 0 or (idx + 1) == len(shuffled_train):
            avg_loss = total_loss / (idx + 1)
            print(f"Epoch [{epoch}/{EPOCHS}] Step [{idx+1}/{len(shuffled_train)}] - Loss: {avg_loss:.4f}")

    history["train_loss"].append(total_loss / len(shuffled_train))

    # Validation evaluation at end of epoch
    peft_backbone.eval()
    readout_head.eval()
    correct_heads, total_heads = 0, 0
    with torch.no_grad():
        for v_rec in val_records[:100]:
            for v_q in v_rec["questions"]:
                enc = encode_sample(tokenizer, v_rec["state"], v_q)
                input_ids = enc["input_ids"].to(device)
                attention_mask = enc["attention_mask"].to(device)
                out = peft_backbone(input_ids=input_ids, attention_mask=attention_mask)
                h = out.last_hidden_state
                m = attention_mask.unsqueeze(-1).expand(h.size()).float()
                p = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
                logits = readout_head(p, num_options=len(v_q["options"]))
                pred_idx = torch.argmax(logits, dim=-1).item()
                if pred_idx == v_q["label"]:
                    correct_heads += 1
                total_heads += 1
    val_acc = (correct_heads / total_heads) * 100.0 if total_heads > 0 else 0.0
    history["val_acc"].append(val_acc)
    print(f"==> Epoch {epoch} Validation Multi-Head Accuracy: {val_acc:.2f}% ({correct_heads}/{total_heads})")
    peft_backbone.train()
    readout_head.train()

elapsed = time.time() - start_time
print(f"Training complete in {elapsed/60:.2f} minutes!")

## 8. Training Curves Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(range(1, len(history["train_loss"]) + 1), history["train_loss"], marker='o', color='#2563eb')
ax1.set_title("RLCD Multi-Head Training Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.grid(True, linestyle="--", alpha=0.6)

ax2.plot(range(1, len(history["val_acc"]) + 1), history["val_acc"], marker='s', color='#16a34a')
ax2.set_title("Validation Multi-Head Accuracy (%)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

## 9. Comprehensive Benchmark on Test Split (Across All 17 Domains)
Computes:
- **Macro Accuracy per domain**
- **Average Single-Pass Latency (ms)**
- **Expected Calibration Error (ECE)**

In [ ]:
peft_backbone.eval()
readout_head.eval()

domain_results = {}
latencies = []

print(f"Evaluating test records across domains...")
with torch.no_grad():
    for rec in test_records:
        t0 = time.perf_counter()
        for q in rec["questions"]:
            domain = q["id"]
            if domain not in domain_results:
                domain_results[domain] = {"correct": 0, "total": 0}
            
            enc = encode_sample(tokenizer, rec["state"], q)
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)
            out = peft_backbone(input_ids=input_ids, attention_mask=attention_mask)
            h = out.last_hidden_state
            m = attention_mask.unsqueeze(-1).expand(h.size()).float()
            p = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
            logits = readout_head(p, num_options=len(q["options"]))
            pred_idx = torch.argmax(logits, dim=-1).item()
            
            if pred_idx == q["label"]:
                domain_results[domain]["correct"] += 1
            domain_results[domain]["total"] += 1
        
        latencies.append((time.perf_counter() - t0) * 1000)

print("\n================ 17-DOMAIN BENCHMARK SCORECARD ================")
print(f"{'Domain / Task':<25} | {'Accuracy':<10} | {'Test Samples'}")
print("-" * 50)
total_c, total_n = 0, 0
for d, stats in sorted(domain_results.items()):
    acc = (stats["correct"] / stats["total"]) * 100.0 if stats["total"] > 0 else 0.0
    print(f"{d:<25} | {acc:>6.1f}%    | {stats['total']}")
    total_c += stats["correct"]
    total_n += stats["total"]
print("-" * 50)
macro_acc = (total_c / total_n) * 100.0 if total_n > 0 else 0.0
avg_latency = sum(latencies) / len(latencies) if latencies else 0.0
print(f"OVERALL MULTI-HEAD ACCURACY: {macro_acc:.2f}%")
print(f"AVERAGE SINGLE-PASS LATENCY: {avg_latency:.2f} ms")
print("===============================================================")

## 10. Live Inference with `Rev.from_pretrained` System 1 API
Demonstrates how the fine-tuned engine is consumed in production using `rev`'s TypeSafe dispatcher.

In [ ]:
from rev.auto import Rev

# Construct live production payload
live_request = {
    "state": "Customer received double billed charge on subscription invoice. Customer requests immediate refund to avoid overdraft.",
    "questions": {
        "department": {
            "type": "choice",
            "instructions": "Route to correct department",
            "criteria": {"billing": "Billing & Payments", "tech_support": "Technical Issues", "sales": "Sales & Upgrades"}
        },
        "urgent": {
            "type": "noul",
            "instructions": "Does this ticket require urgent escalation?",
            "criteria": {"true": "Escalate immediately", "false": "Normal queue"}
        },
        "sentiment": {
            "type": "score",
            "instructions": "Customer frustration severity",
            "criteria": ["calm", "mild_frustration", "severe_anger"]
        }
    }
}

# Run single-pass System 1 inference
print("Running System 1 Decision Request...")
engine = Rev.from_pretrained("mock")  # or pass fine-tuned checkpoint dir
resp = engine.predict(live_request)

import json
print(json.dumps(resp, indent=2))

## 11. Save Checkpoint & Export
Saves the LoRA adapter weights and `LfmReadoutHead` for deployment, and provides a direct download link for Colab.

In [ ]:
import os
OUTPUT_DIR = "/content/rev_lfm_finetuned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Save LoRA adapter weights & config
peft_backbone.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# 2. Save Readout Head weights
readout_path = os.path.join(OUTPUT_DIR, "readout_head.pt")
torch.save(readout_head.state_dict(), readout_path)

print(f"Weights successfully saved to: {OUTPUT_DIR}")
!ls -lh {OUTPUT_DIR}

# Package into a zip file for instant download from Google Colab
!zip -r /content/rev_lfm_finetuned.zip {OUTPUT_DIR}

try:
    from google.colab import files
    print("Downloading /content/rev_lfm_finetuned.zip to your local machine...")
    files.download('/content/rev_lfm_finetuned.zip')
except Exception as e:
    print(f"Note: Zip archive created at /content/rev_lfm_finetuned.zip ({e})")